# MedCompress — ISIC 2020 Melanoma Classification
**Author: Abhishek Shekhar**

Trains EfficientNetB0 teacher + MobileNetV3-Small student with QAT and KD on ISIC 2020.

**Experiments:** Baseline → QAT INT8/FP16 → Student Scratch → KD → KD+QAT

> All outputs auto-save to Google Drive. Re-run any cell safely — completed experiments are skipped.

In [ ]:
import os, sys, time, random, json, warnings
warnings.filterwarnings("ignore")
import numpy as np

# ── Mount Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── GPU check ───────────────────────────────────────────────────────────────
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError("No GPU! Go to Runtime → Change runtime type → T4 GPU")
tf.config.set_visible_devices(gpus[0], 'GPU')
tf.config.experimental.set_memory_growth(gpus[0], True)
print(f"TensorFlow {tf.__version__}  GPU: {gpus[0].name}")

# ── Directories (all outputs persist on Drive) ──────────────────────────────
EXP_NAME    = "isic_classification"   # ← overridden per notebook
DRIVE_BASE  = f"/content/drive/MyDrive/medcompress/isic_classification"
DATA_DIR    = f"/content/data/isic_classification"
RESULTS_DIR = f"{DRIVE_BASE}/results"
CKPT_DIR    = f"{DRIVE_BASE}/checkpoints"
MODELS_DIR  = f"{DRIVE_BASE}/models"
TFLITE_DIR  = f"{DRIVE_BASE}/tflite"
for d in [DRIVE_BASE, DATA_DIR, RESULTS_DIR, CKPT_DIR, MODELS_DIR, TFLITE_DIR]:
    os.makedirs(d, exist_ok=True)

SESSION_START = time.time()
print(f"Drive base : {DRIVE_BASE}")
print("Setup complete ✓")

In [ ]:
%%capture install_out
!pip install -q tensorflow-model-optimization tf2onnx onnx scikit-learn pillow tqdm
import tensorflow_model_optimization as tfmot
import tf2onnx
print("All packages installed ✓")

In [ ]:
# ── Kaggle credentials (run once per session) ──────────────────────────────
import os
KAGGLE_JSON = os.path.expanduser("~/.kaggle/kaggle.json")
if not os.path.exists(KAGGLE_JSON):
    from google.colab import files
    print("Upload your kaggle.json API token (from kaggle.com → Settings → API):")
    files.upload()
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    !mv kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle credentials saved ✓")
else:
    print("Kaggle credentials already present ✓")

In [ ]:
# ── Download ISIC 2020 ──────────────────────────────────────────────────────
ISIC_DIR = f"{DATA_DIR}/isic2020"
os.makedirs(ISIC_DIR, exist_ok=True)

if not os.path.exists(f"{ISIC_DIR}/train.csv"):
    print("Downloading ISIC 2020 (this may take ~10 min)...")
    !kaggle competitions download -c siim-isic-melanoma-classification -p {ISIC_DIR}
    !cd {ISIC_DIR} && unzip -q "*.zip" && rm -f *.zip
    print("Download complete ✓")
else:
    print(f"Data cached at {ISIC_DIR} ✓")

import pandas as pd
df = pd.read_csv(f"{ISIC_DIR}/train.csv")
print(f"Dataset: {len(df)} samples  |  Melanoma: {df['target'].mean()*100:.2f}%")

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────
IMG_SIZE   = 224
BATCH_SIZE = 32
SEEDS      = [42, 123, 456, 789, 1024]

CFG = {
    "baseline":       {"epochs":20, "lr":1e-4, "patience":7},
    "qat":            {"epochs":10, "lr":1e-5},
    "student_scratch":{"epochs":25, "lr":1e-4, "patience":10},
    "kd":             {"epochs":25, "lr":1e-4, "patience":10, "temperature":4.0, "alpha":0.7},
    "kd_qat":         {"epochs":10, "lr":1e-5},
}

# Find image directory
jpeg_dir = f"{ISIC_DIR}/jpeg/train"
if not os.path.exists(jpeg_dir): jpeg_dir = f"{ISIC_DIR}/train"
if not os.path.exists(jpeg_dir):
    for root,dirs,files in os.walk(ISIC_DIR):
        if len([f for f in files if f.endswith(".jpg")])>100:
            jpeg_dir=root; break

df["image_path"] = df["image_name"].apply(lambda x: f"{jpeg_dir}/{x}.jpg")
df = df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)
counts = df["target"].value_counts()
CLASS_WEIGHTS = {0: len(df)/(2*counts[0]), 1: len(df)/(2*counts[1])}
print(f"Images found: {len(df)}  |  Class weights: {CLASS_WEIGHTS}")

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

def compute_cls_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob.ravel() >= thr).astype(int)
    y_true = y_true.astype(int).ravel()
    auc  = roc_auc_score(y_true, y_prob.ravel())
    f1   = f1_score(y_true, y_pred, zero_division=0)
    sens = recall_score(y_true, y_pred, zero_division=0)
    tn,fp,fn,tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    spec = tn/(tn+fp) if (tn+fp)>0 else 0.0
    return {"auc":float(auc),"f1":float(f1),"sensitivity":float(sens),"specificity":float(spec)}

def export_tflite(model, path, quant="fp32", calib_ds=None, n_calib=200):
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    if quant == "int8":
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        if calib_ds:
            samples = []
            for imgs, _ in calib_ds:
                for i in range(len(imgs)):
                    samples.append(imgs[i].numpy())
                    if len(samples) >= n_calib: break
                if len(samples) >= n_calib: break
            def rep():
                for s in samples: yield [np.expand_dims(s,0)]
            conv.representative_dataset = rep
            conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
            conv.inference_input_type  = tf.uint8
            conv.inference_output_type = tf.uint8
    elif quant == "fp16":
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.target_spec.supported_types = [tf.float16]
    data = conv.convert()
    with open(path,"wb") as f: f.write(data)
    mb = os.path.getsize(path)/1e6
    print(f"  TFLite {quant}: {os.path.basename(path)} ({mb:.1f} MB)")
    return mb

def eval_tflite(path, dataset, warmup=5):
    interp = tf.lite.Interpreter(model_path=path, num_threads=4)
    interp.allocate_tensors()
    inp_d = interp.get_input_details()[0]
    out_d = interp.get_output_details()[0]
    preds, labels, lats = [], [], []
    for imgs, lbls in dataset:
        for i in range(len(imgs)):
            x = np.expand_dims(imgs[i].numpy(), 0)
            if inp_d["dtype"] == np.uint8:
                sc,zp = inp_d["quantization"]; x=(x/sc+zp).astype(np.uint8)
            interp.set_tensor(inp_d["index"], x)
            t0=time.perf_counter(); interp.invoke(); t1=time.perf_counter()
            lats.append((t1-t0)*1000)
            r=interp.get_tensor(out_d["index"])
            if out_d["dtype"]==np.uint8:
                sc,zp=out_d["quantization"]; r=(r.astype(np.float32)-zp)*sc
            preds.append(r.squeeze()); labels.append(lbls[i].numpy())
    lats = np.array(lats[warmup:])
    m = compute_cls_metrics(np.array(labels), np.array(preds))
    m["latency_ms"] = float(np.median(lats))
    m["latency_p95_ms"] = float(np.percentile(lats,95))
    m["size_mb"] = os.path.getsize(path)/1e6
    print(f"  TFLite AUC={m['auc']:.4f}  lat={m['latency_ms']:.1f}ms  sz={m['size_mb']:.1f}MB")
    return m

def done_flag(name): return f"{RESULTS_DIR}/{name}.done"
def is_done(name): return os.path.exists(done_flag(name))
def mark_done(name, result):
    with open(f"{RESULTS_DIR}/{name}.json","w") as f: json.dump(result, f)
    open(done_flag(name),"w").close()
def load_result(name):
    with open(f"{RESULTS_DIR}/{name}.json") as f: return json.load(f)

def train_model(model, train_ds, val_ds, cfg, name, monitor="val_auc", mode="max"):
    """Train with Drive checkpoint + resume. Returns history."""
    ckpt = f"{CKPT_DIR}/{name}_best.keras"
    epoch_f = f"{CKPT_DIR}/{name}_epoch.json"
    initial_epoch = 0
    if os.path.exists(ckpt):
        try:
            model.load_weights(ckpt)
            if os.path.exists(epoch_f):
                initial_epoch = json.load(open(epoch_f)).get("epoch",0)
            print(f"  Resumed from epoch {initial_epoch} ✓")
        except Exception as e:
            print(f"  Checkpoint load failed ({e}), starting fresh")
    class _EpochLog(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            json.dump({"epoch":epoch+1}, open(epoch_f,"w"))
    cbs = [
        tf.keras.callbacks.ModelCheckpoint(
            ckpt, save_best_only=True, monitor=monitor, mode=mode, verbose=0),
        tf.keras.callbacks.EarlyStopping(
            patience=cfg.get("patience",7), monitor=monitor, mode=mode,
            restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor=monitor, factor=0.5, patience=3, mode=mode, verbose=0),
        _EpochLog()
    ]
    history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=cfg.get("epochs",20), initial_epoch=initial_epoch,
        class_weight=cfg.get("class_weight"), callbacks=cbs, verbose=1)
    return model, history

print("Utilities loaded ✓")

In [ ]:
# ── Data pipeline ───────────────────────────────────────────────────────────
def make_ds(dataframe, augment=False, shuffle=False):
    paths  = dataframe["image_path"].values
    labels = dataframe["target"].values.astype(np.float32)
    def load(path, label):
        img = tf.image.decode_jpeg(tf.io.read_file(path), channels=3)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img = tf.cast(img, tf.float32)/127.5 - 1.0
        return img, label
    def aug(img, label):
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.random_brightness(img, 0.2)
        img = tf.image.random_contrast(img, 0.8, 1.2)
        return img, label
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle: ds = ds.shuffle(len(paths), reshuffle_each_iteration=True)
    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    if augment: ds = ds.map(aug, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_df, test_df = train_test_split(df, test_size=0.15, stratify=df["target"], random_state=42)
train_df, val_df  = train_test_split(train_df, test_size=0.15/0.85, stratify=train_df["target"], random_state=42)
print(f"Splits: train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
train_ds = make_ds(train_df, augment=True, shuffle=True)
val_ds   = make_ds(val_df)
test_ds  = make_ds(test_df)

In [ ]:
# ── Model builders ──────────────────────────────────────────────────────────
def build_teacher():
    bb = keras.applications.EfficientNetB0(include_top=False,weights="imagenet",
                                            input_shape=(IMG_SIZE,IMG_SIZE,3))
    for l in bb.layers[:-20]: l.trainable=False
    x = inputs = keras.Input((IMG_SIZE,IMG_SIZE,3))
    x = bb(inputs,training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256,activation="relu")(x)
    x = layers.Dropout(0.15)(x)
    return keras.Model(inputs, layers.Dense(1,activation="sigmoid")(x), name="efficientnetb0")

def build_student():
    bb = keras.applications.MobileNetV3Small(include_top=False,weights="imagenet",
                                              input_shape=(IMG_SIZE,IMG_SIZE,3))
    for l in bb.layers[:-10]: l.trainable=False
    x = inputs = keras.Input((IMG_SIZE,IMG_SIZE,3))
    x = bb(inputs,training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128,activation="relu")(x)
    x = layers.Dropout(0.15)(x)
    return keras.Model(inputs, layers.Dense(1,activation="sigmoid")(x), name="mobilenetv3s")

def eval_model(model, ds, tag=""):
    preds,lbls = [],[]
    for imgs,labs in ds:
        preds.append(model(imgs,training=False).numpy()); lbls.append(labs.numpy())
    m = compute_cls_metrics(np.concatenate(lbls), np.concatenate(preds))
    print(f"  [{tag}] AUC={m['auc']:.4f}  Sens={m['sensitivity']:.4f}  "
          f"Spec={m['specificity']:.4f}  F1={m['f1']:.4f}")
    return m

print("Models defined ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 1 — Baseline EfficientNetB0 (multi-seed)
# ══════════════════════════════════════════════════════════════════════════════
baseline_results = []
for seed in SEEDS:
    key = f"baseline_s{seed}"
    if is_done(key):
        print(f"  ⏩ {key} already done"); baseline_results.append(load_result(key)); continue
    print(f"\n── Baseline seed={seed} ──")
    set_seed(seed)
    m = build_teacher()
    m.compile(optimizer=keras.optimizers.Adam(CFG["baseline"]["lr"]),
              loss="binary_crossentropy", metrics=[keras.metrics.AUC(name="auc")])
    cfg = dict(CFG["baseline"]); cfg["class_weight"] = CLASS_WEIGHTS
    m, _ = train_model(m, train_ds, val_ds, cfg, key, monitor="val_auc")
    metrics = eval_model(m, test_ds, key)
    m.save(f"{MODELS_DIR}/efficientnetb0_s{seed}.keras")
    mark_done(key, metrics); baseline_results.append(metrics)

print("\n── Baseline Summary (mean ± std over 5 seeds) ──")
for k in ["auc","sensitivity","specificity","f1"]:
    v=[r[k] for r in baseline_results]
    print(f"  {k}: {np.mean(v):.4f} ± {np.std(v):.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 2 — QAT INT8 + FP16
# ══════════════════════════════════════════════════════════════════════════════
import tensorflow_model_optimization as tfmot
qat_int8_results, qat_fp16_results = [], []

for seed in SEEDS:
    key_i8 = f"qat_int8_s{seed}"; key_fp16 = f"qat_fp16_s{seed}"
    if is_done(key_i8) and is_done(key_fp16):
        print(f"  ⏩ QAT s{seed} done")
        qat_int8_results.append(load_result(key_i8))
        qat_fp16_results.append(load_result(key_fp16)); continue
    print(f"\n── QAT seed={seed} ──")
    set_seed(seed)
    base = keras.models.load_model(f"{MODELS_DIR}/efficientnetb0_s{seed}.keras")
    qat_m = tfmot.quantization.keras.quantize_model(base)
    qat_m.compile(optimizer=keras.optimizers.Adam(CFG["qat"]["lr"]),
                  loss="binary_crossentropy", metrics=[keras.metrics.AUC(name="auc")])
    qat_m.fit(train_ds, validation_data=val_ds, epochs=CFG["qat"]["epochs"],
              class_weight=CLASS_WEIGHTS, verbose=1)
    stripped = tfmot.quantization.keras.strip_pruning(qat_m)
    if not is_done(key_i8):
        p = f"{TFLITE_DIR}/effb0_int8_s{seed}.tflite"
        export_tflite(stripped, p, "int8", train_ds)
        m = eval_tflite(p, test_ds); mark_done(key_i8, m)
        qat_int8_results.append(m)
    if not is_done(key_fp16):
        p = f"{TFLITE_DIR}/effb0_fp16_s{seed}.tflite"
        export_tflite(stripped, p, "fp16")
        m = eval_tflite(p, test_ds); mark_done(key_fp16, m)
        qat_fp16_results.append(m)

for label,res in [("QAT INT8",qat_int8_results),("QAT FP16",qat_fp16_results)]:
    print(f"\n── {label} ──")
    for k in ["auc","latency_ms","size_mb"]:
        if k in res[0]: v=[r[k] for r in res]; print(f"  {k}: {np.mean(v):.4f} ± {np.std(v):.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 3 — Student from Scratch + Teacher for KD
# ══════════════════════════════════════════════════════════════════════════════
scratch_results = []
for seed in SEEDS:
    key = f"scratch_s{seed}"
    if is_done(key):
        print(f"  ⏩ {key} done"); scratch_results.append(load_result(key)); continue
    print(f"\n── Student scratch seed={seed} ──")
    set_seed(seed)
    s = build_student()
    s.compile(optimizer=keras.optimizers.Adam(CFG["student_scratch"]["lr"]),
              loss="binary_crossentropy", metrics=[keras.metrics.AUC(name="auc")])
    cfg = dict(CFG["student_scratch"]); cfg["class_weight"]=CLASS_WEIGHTS
    s, _ = train_model(s, train_ds, val_ds, cfg, key, monitor="val_auc")
    m = eval_model(s, test_ds, key); s.save(f"{MODELS_DIR}/student_scratch_s{seed}.keras")
    mark_done(key, m); scratch_results.append(m)

# Train teacher (single seed=42) for KD
teacher_path = f"{MODELS_DIR}/teacher_for_kd.keras"
if not os.path.exists(teacher_path):
    print("\n── Training teacher (EfficientNetB0 s42) for KD ──")
    set_seed(42)
    teacher = keras.models.load_model(f"{MODELS_DIR}/efficientnetb0_s42.keras")
    teacher.save(teacher_path)
teacher = keras.models.load_model(teacher_path)
print(f"Teacher loaded ✓  AUC ref: {eval_model(teacher, test_ds, 'teacher')['auc']:.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 4 — Knowledge Distillation
# ══════════════════════════════════════════════════════════════════════════════
T = CFG["kd"]["temperature"]
ALPHA = CFG["kd"]["alpha"]
kd_results = []

for seed in SEEDS:
    key = f"kd_s{seed}"
    if is_done(key):
        print(f"  ⏩ {key} done"); kd_results.append(load_result(key)); continue
    print(f"\n── KD seed={seed} (T={T}, α={ALPHA}) ──")
    set_seed(seed)
    student = build_student()
    opt = keras.optimizers.Adam(CFG["kd"]["lr"])
    best_auc, patience_count = 0, 0
    ckpt_path = f"{CKPT_DIR}/kd_s{seed}.keras"
    if os.path.exists(ckpt_path): student.load_weights(ckpt_path); print("  Resumed ✓")
    for epoch in range(CFG["kd"]["epochs"]):
        ep_loss, n = 0, 0
        for imgs, lbls in train_ds:
            with tf.GradientTape() as tape:
                t_out = teacher(imgs, training=False)
                s_out = student(imgs, training=True)
                def soft(x): return tf.nn.sigmoid(tf.math.log(x/(1-x+1e-7)+1e-7)/T)
                ts, ss = soft(t_out), soft(s_out)
                eps=1e-7
                kl = ts*tf.math.log((ts+eps)/(ss+eps)) + (1-ts)*tf.math.log((1-ts+eps)/(1-ss+eps))
                loss = ALPHA*tf.reduce_mean(kl)*(T**2) + (1-ALPHA)*tf.reduce_mean(
                    keras.losses.binary_crossentropy(tf.expand_dims(lbls,-1), s_out))
            grads = tape.gradient(loss, student.trainable_variables)
            opt.apply_gradients(zip(grads, student.trainable_variables))
            ep_loss+=loss.numpy(); n+=1
        val_auc = eval_model(student, val_ds, f"ep{epoch+1}")["auc"]
        print(f"  Epoch {epoch+1}: loss={ep_loss/n:.4f}  val_auc={val_auc:.4f}")
        if val_auc > best_auc:
            best_auc=val_auc; patience_count=0; student.save_weights(ckpt_path)
        else:
            patience_count+=1
            if patience_count >= CFG["kd"]["patience"]: print("  Early stop"); break
    student.load_weights(ckpt_path)
    m = eval_model(student, test_ds, key)
    student.save(f"{MODELS_DIR}/student_kd_s{seed}.keras")
    mark_done(key, m); kd_results.append(m)

print("\n── KD Summary ──")
for k in ["auc","sensitivity","specificity","f1"]:
    v=[r[k] for r in kd_results]; print(f"  {k}: {np.mean(v):.4f} ± {np.std(v):.4f}")
scratch_auc = np.mean([r["auc"] for r in scratch_results])
kd_auc      = np.mean([r["auc"] for r in kd_results])
print(f"\n★ Distillation gain: {(kd_auc-scratch_auc)*100:+.2f}% AUC")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 5 — KD + QAT INT8 (compressed endpoint model)
# ══════════════════════════════════════════════════════════════════════════════
kd_qat_results = []
for seed in SEEDS:
    key = f"kd_qat_s{seed}"
    if is_done(key):
        print(f"  ⏩ {key} done"); kd_qat_results.append(load_result(key)); continue
    print(f"\n── KD+QAT seed={seed} ──")
    set_seed(seed)
    base = keras.models.load_model(f"{MODELS_DIR}/student_kd_s{seed}.keras")
    qat_m = tfmot.quantization.keras.quantize_model(base)
    qat_m.compile(optimizer=keras.optimizers.Adam(1e-5),
                  loss="binary_crossentropy", metrics=[keras.metrics.AUC(name="auc")])
    qat_m.fit(train_ds, validation_data=val_ds, epochs=10,
              class_weight=CLASS_WEIGHTS, verbose=1)
    stripped = tfmot.quantization.keras.strip_pruning(qat_m)
    p = f"{TFLITE_DIR}/student_kd_qat_int8_s{seed}.tflite"
    export_tflite(stripped, p, "int8", train_ds)
    m = eval_tflite(p, test_ds); mark_done(key, m); kd_qat_results.append(m)

print("\n── KD+QAT INT8 Summary ──")
for k in ["auc","latency_ms","size_mb"]:
    v=[r[k] for r in kd_qat_results]; print(f"  {k}: {np.mean(v):.4f} ± {np.std(v):.4f}")

# Save best model to Drive root for endpoint
import shutil
best_s42 = f"{TFLITE_DIR}/student_kd_qat_int8_s42.tflite"
if os.path.exists(best_s42):
    shutil.copy(best_s42, f"{TFLITE_DIR}/mobilenetv3s_kd_qat_int8.tflite")

In [ ]:
# ── Final results table → Drive ──────────────────────────────────────────────
def summarize(results, name):
    row = {"method":name}
    for k in ["auc","sensitivity","specificity","f1"]:
        v=[r[k] for r in results]; row[f"{k}_mean"]=round(np.mean(v),4); row[f"{k}_std"]=round(np.std(v),4)
    for k in ["latency_ms","size_mb"]:
        if results and k in results[0]: v=[r[k] for r in results]; row[k]=round(np.mean(v),3)
    return row

table = pd.DataFrame([
    summarize(baseline_results,  "Baseline FP32"),
    summarize(qat_int8_results,  "QAT INT8"),
    summarize(qat_fp16_results,  "QAT FP16"),
    summarize(scratch_results,   "Student Scratch"),
    summarize(kd_results,        "KD (T=4 α=0.7)"),
    summarize(kd_qat_results,    "KD+QAT INT8"),
])
table.to_csv(f"{RESULTS_DIR}/isic_full_results.csv", index=False)
print("Results saved to Drive ✓")
print(table[["method","auc_mean","auc_std","sensitivity_mean","specificity_mean","f1_mean"]].to_string(index=False))
print(f"\nAll files at: {DRIVE_BASE}")

In [ ]:
# ─── Endpoint inference wrapper ────────────────────────────────────────────
class MedCompressEndpoint:
    """Minimal TFLite inference endpoint — drop-in for real deployment."""
    def __init__(self, tflite_path):
        self.interp = tf.lite.Interpreter(model_path=tflite_path, num_threads=4)
        self.interp.allocate_tensors()
        self.inp = self.interp.get_input_details()[0]
        self.out = self.interp.get_output_details()[0]
        self.size_mb = os.path.getsize(tflite_path)/1e6
        print(f"Endpoint loaded: {os.path.basename(tflite_path)} ({self.size_mb:.1f} MB)")

    def preprocess(self, image_array):
        """Accepts uint8 HWC image, returns model-ready input."""
        import cv2
        h,w = self.inp["shape"][1], self.inp["shape"][2]
        img = cv2.resize(image_array.astype(np.float32), (w,h))
        img = img / 127.5 - 1.0
        if self.inp["dtype"] == np.uint8:
            sc,zp = self.inp["quantization"]
            img = ((img + 1.0)*127.5 / sc + zp).astype(np.uint8)
        return np.expand_dims(img, 0)

    def predict(self, image_array):
        x = self.preprocess(image_array)
        self.interp.set_tensor(self.inp["index"], x)
        self.interp.invoke()
        r = self.interp.get_tensor(self.out["index"])
        if self.out["dtype"] == np.uint8:
            sc,zp = self.out["quantization"]
            r = (r.astype(np.float32)-zp)*sc
        return float(r.squeeze())

    def benchmark(self, n=100):
        dummy = np.random.randint(0,255,(224,224,3),dtype=np.uint8)
        times = []
        for _ in range(n):
            t0=time.perf_counter(); self.predict(dummy); t1=time.perf_counter()
            times.append((t1-t0)*1000)
        times = np.array(times[5:])
        print(f"  Latency: median={np.median(times):.1f}ms  p95={np.percentile(times,95):.1f}ms")
        return {"median_ms":float(np.median(times)),"p95_ms":float(np.percentile(times,95))}

# Test endpoint with best KD+QAT model if available
kd_qat_path = f"{TFLITE_DIR}/mobilenetv3s_kd_qat_int8.tflite"
if os.path.exists(kd_qat_path):
    ep = MedCompressEndpoint(kd_qat_path)
    bench = ep.benchmark()
    print(f"\nEndpoint ready for deployment ✓")
    print(f"To deploy as API: see deploy/cli.py in the repo")
else:
    print("Run experiments first to generate TFLite models")